# SciLite Ground Truth Generator

Fetches **Europe PMC SciLite annotations** for a set of PMC articles and builds a ground-truth dataset that can be used to evaluate the data-gatherer pipeline.

SciLite annotations cover entity types such as:
- **Accession Numbers** — dataset/sequence accessions (GEO, UniProt, PDB, …)
- **Gene / Protein** mentions
- **Disease**, **Chemical**, **Organism** mentions
- **Gene Ontology** terms

The accession-number annotations are the primary ground truth for the dataset-citation extraction task.

**API limit:** 8 article IDs per request.  
**Script:** `scripts/scrape_annotations.py` (all fetch logic lives there — this notebook just orchestrates and explores).

In [1]:
import sys, json, logging
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path().resolve()))  # project root

from scripts.scrape_annotations import (
    load_ids_from_csv,
    make_session,
    fetch_annotations,
    load_checkpoint,
    extract_accession_numbers,
    to_dataframe,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

## Configuration

In [2]:
# --- Input ---
PMC_ID_FILE   = "k8s/input/article_ids_REV_pmc.csv"   # 118 k articles
EVAL_ID_FILE  = "k8s/input/article_ids_eval.csv"       # smaller eval set

# --- Output ---
OUTPUT_DIR    = Path("scripts/output")
OUTPUT_SUBSET = OUTPUT_DIR / "annotations_subset.json"
OUTPUT_FULL   = OUTPUT_DIR / "annotations_full.json"
OUTPUT_GT_CSV = OUTPUT_DIR / "scilite_ground_truth.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Fetch params ---
BATCH_SIZE = 8    # hard API limit
PAUSE      = 0  # seconds between requests

## 1. Load PMC IDs

In [3]:
pmc_ids = load_ids_from_csv(PMC_ID_FILE)
print(f"Loaded {len(pmc_ids):,} PMC IDs from {PMC_ID_FILE}")
print("Sample:", pmc_ids[:5])

Loaded 118,575 PMC IDs from k8s/input/article_ids_REV_pmc.csv
Sample: ['PMC11895760', 'PMC11773904', 'PMC11398498', 'PMC11061317', 'PMC11402562']


## 2. Fetch Annotations

### 2a. Subset run (quick test)

In [4]:
SUBSET_N = 16
subset   = pmc_ids[:SUBSET_N]

session = make_session()
done = fetch_annotations(subset, session, batch_size=BATCH_SIZE, pause=PAUSE, output=OUTPUT_SUBSET)

with OUTPUT_SUBSET.open("w") as fh:
    json.dump(done, fh, indent=2)

print(f"\nFetched {len(done)} articles → {OUTPUT_SUBSET}")
for pmcid, anns in done.items():
    print(f"  {pmcid}: {len(anns):>4d} annotations")

2026-05-15 15:07:26,340 INFO Total IDs: 16  |  already done: 0  |  to fetch: 16  |  batches: 2
2026-05-15 15:07:26,341 INFO [1/2] Fetching 8 IDs
2026-05-15 15:07:28,129 INFO [2/2] Fetching 8 IDs



Fetched 16 articles → scripts/output/annotations_subset.json
  PMC6826127:    1 annotations
  PMC10328412:    0 annotations
  PMC11061317:    2 annotations
  PMC11398498:   17 annotations
  PMC11402562:    1 annotations
  PMC11773904:    0 annotations
  PMC11790920:    1 annotations
  PMC11895760:   10 annotations
  PMC9721094:    3 annotations
  PMC11424477:   12 annotations
  PMC11472906:    2 annotations
  PMC11585809:    1 annotations
  PMC11816699:    0 annotations
  PMC11827837:   34 annotations
  PMC11890945:    6 annotations
  PMC11463496:    0 annotations


### 2b. Full run (118 k articles, resumable)

Estimated time at batch=8 / pause=0.2 s: **~15 hours**.  
Interrupt and re-run freely — the checkpoint file resumes automatically.

In [ ]:
done_full = load_checkpoint(OUTPUT_FULL)  # resumes from prior run if present
print(f"Checkpoint: {len(done_full):,} articles already fetched")

session = make_session()
done_full = fetch_annotations(
    pmc_ids, session,
    batch_size=BATCH_SIZE, pause=PAUSE,
    done=done_full, output=OUTPUT_FULL,
)

with OUTPUT_FULL.open("w") as fh:
    json.dump(done_full, fh, indent=2)
print(f"Done. {len(done_full):,} articles → {OUTPUT_FULL}")

## 3. Explore Results

Run on the subset output for a quick sanity-check; swap `done` → `done_full` for the full dataset.

In [ ]:
# Load subset results (change to OUTPUT_FULL for full dataset)
with OUTPUT_SUBSET.open() as fh:
    data = json.load(fh)

total_articles    = len(data)
articles_with_ann = sum(1 for v in data.values() if v)
total_annotations = sum(len(v) for v in data.values())

print(f"Articles:             {total_articles:>6,}")
print(f"  with annotations:   {articles_with_ann:>6,}  ({articles_with_ann/total_articles:.0%})")
print(f"Total annotations:    {total_annotations:>6,}")
print(f"Avg per article:      {total_annotations/max(articles_with_ann,1):>6.1f}")

In [ ]:
from collections import Counter

type_counter = Counter()
for anns in data.values():
    for ann in anns:
        for tag in ann.get("tags", []):
            uri = tag.get("uri", "")
            # Derive a short type label from the URI
            if "europepmc" in uri or "accessionNumber" in uri.lower():
                type_counter["Accession Number"] += 1
            elif "GO:" in uri:
                type_counter["Gene Ontology"] += 1
            elif "uniprot" in uri:
                type_counter["UniProt"] += 1
            elif "ncbi" in uri and "gene" in uri:
                type_counter["Gene/Protein"] += 1
            elif "disease" in uri.lower() or "DOID" in uri:
                type_counter["Disease"] += 1
            elif "chebi" in uri.lower() or "pubchem" in uri.lower():
                type_counter["Chemical"] += 1
            elif "ncbitaxon" in uri.lower() or "taxonomy" in uri.lower():
                type_counter["Organism"] += 1
            else:
                type_counter["Other"] += 1

print("Annotation types:")
for label, count in type_counter.most_common():
    print(f"  {label:<22} {count:>6,}")

In [ ]:
# Show a few raw annotations from the first article that has some
for pmcid, anns in data.items():
    if anns:
        print(f"{pmcid}  ({len(anns)} annotations — showing first 5)")
        for ann in anns[:5]:
            print(f"  exact={ann.get('exact')!r:30s}  section={ann.get('section','?')!r}")
            for tag in ann.get("tags", []):
                print(f"    → {tag.get('name')!r}  uri={tag.get('uri','')[:60]}")
        break

## 4. Export as Ground Truth CSV

Flatten the JSON into a tidy table with one row per annotation.  
Filter to **Accession Number** annotations — these are the direct ground truth for dataset-citation evaluation.

| column | description |
|---|---|
| `pmcid` | Source article |
| `exact` | Accession string as it appears in the text |
| `section` | Article section (Abstract, Methods, …) |
| `tag_name` | Human-readable label from SciLite |
| `tag_uri` | Ontology URI |
| `prefix` / `postfix` | Surrounding text (useful for context) |

In [ ]:
with OUTPUT_SUBSET.open() as fh:
    data = json.load(fh)

acc = extract_accession_numbers(data)
gt  = to_dataframe(acc)

print(f"Articles with accession numbers: {sum(1 for v in acc.values() if v)} / {len(acc)}")
print(f"Total accession rows:            {len(gt)}")

gt.to_csv(OUTPUT_GT_CSV, index=False)
print(f"Saved → {OUTPUT_GT_CSV}")
gt.head(10)